In [1]:
import pandas as pd
from pathlib import Path
import os

In [26]:
df = pd.read_csv("../data/main/edc_all_artists.csv")
df.drop_duplicates(inplace=True)
df.head()

,year,artist
0,2024,1080p
1,2024,A Shade of Black
2,2024,Aaron K
3,2024,Abana
4,2024,D. Zeledon


In [35]:
if "year" in df.columns:
    df = df.drop(columns=["year"])

# Drop duplicates after removing the year column
df = df.drop_duplicates()
print(f"Unique artists count: {len(df)}")

Unique artists count: 1046


In [36]:
# Save the deduplicated data
df.to_csv("../data/main/edc_artists_no_duplicates.csv", index=False)
print("Saved to ../data/main/edc_artists_no_duplicates.csv")

Saved to ../data/main/edc_artists_no_duplicates.csv


In [37]:
df.count()


artist    1046
dtype: int64

Clean artists and their stats csv

In [3]:
artists_stats = pd.read_csv("../data/main/edc_artist_stats.csv")
artists_stats.head()

,Artist,Followers,Streams,Playlists,Playlist Reach,Charts,Shazams,Videos,Views,DJ Supports,Error
0,1080p,6,5986,1,641,NaN,17,45,9800,0,NaN
1,A Shade of Black,NaN,15.1K\n,NaN,11K\n,7\n,NaN,NaN,NaN,0,Locator.click: Timeout 30000ms exceeded.\nCall...
2,Aaron K,1,2163,NaN,NaN,1.0,4,2,NaN,0,NaN
3,Abana,7710,468K,175,4.22M,60.0,15.3K,110,1.89M,93,NaN
4,D. Zeledon,5776,69.6K,25,218K,18.0,764,31,9672,58,NaN


In [4]:
artists_stats = artists_stats.drop(columns=["Error"])
artists_stats.head()    

,Artist,Followers,Streams,Playlists,Playlist Reach,Charts,Shazams,Videos,Views,DJ Supports
0,1080p,6,5986,1,641,NaN,17,45,9800,0
1,A Shade of Black,NaN,15.1K\n,NaN,11K\n,7\n,NaN,NaN,NaN,0
2,Aaron K,1,2163,NaN,NaN,1.0,4,2,NaN,0
3,Abana,7710,468K,175,4.22M,60.0,15.3K,110,1.89M,93
4,D. Zeledon,5776,69.6K,25,218K,18.0,764,31,9672,58


In [8]:
# Remove tabs/newlines and trailing " NaN ..." fragments from text cells
str_cols = artists_stats.select_dtypes(include=["object"]).columns
artists_stats[str_cols] = artists_stats[str_cols].replace({r"[\t\r\n]+": " "}, regex=True)
artists_stats[str_cols] = artists_stats[str_cols].replace({r"\s+NaN.*$": ""}, regex=True)
artists_stats[str_cols] = artists_stats[str_cols].apply(lambda s: s.str.strip())

# Replace NaN with 0
artists_stats = artists_stats.fillna(0)

artists_stats.head()

,Artist,Followers,Streams,Playlists,Playlist Reach,Charts,Shazams,Videos,Views,DJ Supports
0,1080p,6,5986,1,641,0,17,45,9800,0
1,A Shade of Black,0,15.1K,0,11K,7,0,0,0,0
2,Aaron K,1,2163,0,0,1.0,4,2,0,0
3,Abana,7710,468K,175,4.22M,60.0,15.3K,110,1.89M,93
4,D. Zeledon,5776,69.6K,25,218K,18.0,764,31,9672,58


In [10]:
# Convert K, M, B suffixes to actual numbers
def convert_suffix(val):
    if pd.isna(val) or val == 0:
        return 0
    val_str = str(val).strip()
    if val_str == '':
        return 0
    try:
        if val_str.endswith('K'):
            return int(float(val_str[:-1]) * 1_000)
        elif val_str.endswith('M'):
            return int(float(val_str[:-1]) * 1_000_000)
        elif val_str.endswith('B'):
            return int(float(val_str[:-1]) * 1_000_000_000)
        else:
            return int(float(val_str))
    except:
        return val

# Columns to convert
cols_to_convert = ['Followers', 'Streams', 'Playlists', 'Playlist Reach', 'Charts', 'Shazams', 'Videos', 'Views', 'DJ Supports']

for col in cols_to_convert:
    if col in artists_stats.columns:
        artists_stats[col] = artists_stats[col].apply(convert_suffix)

artists_stats.head()

,Artist,Followers,Streams,Playlists,Playlist Reach,Charts,Shazams,Videos,Views,DJ Supports
0,1080p,6,5986,1,641,0,17,45,9800,0
1,A Shade of Black,0,15100,0,11000,7,0,0,0,0
2,Aaron K,1,2163,0,0,1,4,2,0,0
3,Abana,7710,468000,175,4220000,60,15300,110,1890000,93
4,D. Zeledon,5776,69600,25,218000,18,764,31,9672,58


In [12]:
# Lowercase all column names
artists_stats.columns = artists_stats.columns.str.lower()

# Normalize artist names (strip and lowercase)
if 'artist' in artists_stats.columns:
    artists_stats['artist'] = artists_stats['artist'].astype(str).str.strip().str.lower()

artists_stats.head()

,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports
0,1080p,6,5986,1,641,0,17,45,9800,0
1,a shade of black,0,15100,0,11000,7,0,0,0,0
2,aaron k,1,2163,0,0,1,4,2,0,0
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58


In [13]:
artists_stats.to_csv("../data/main/edc_artist_stats_cleaned.csv", index=False)